<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Datos Atípicos: los Costales con Piedras de la Plaza 🥔 (Para Dummies)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/07%20-%20Temas%20Avanzados%20en%20Data%20Analysis/Para%20Dummies/05_Atipicos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 05 del Módulo 07 (Detección de datos atípicos y simulación multivariada)](../05_Deteccion_de_Datos_Atipicos_Simulacion_Multivariada.ipynb). Un **dato atípico** es como el **costal «raro» en la plaza de mercado**: uno que pesa mucho más que los demás porque el vendedor le metió piedras. La báscula de la plaza ya no es lo que era: ¿cómo **descubrir** a los tramposos sin acusar a los honestos? Y una pregunta más difícil: **¿cómo sabemos que nuestro método de descubrir funciona?** La respuesta es **practicar con trampas que nosotros mismos sembramos**, para saber cuántas encuentra y cuántas veces se equivoca.

| En la plaza | En estadística |
|---|---|
| El costal con piedras | Un **dato atípico** |
| «Raro = lejos del promedio» (el vendedor tramposo **sube el promedio**) | La regla del promedio, que **se deja engañar** |
| «Raro = lejos del valor del medio» | La regla de la **mediana**, que **resiste** |
| Un niño de 1,90 m y 45 kg: alto y liviano, **raro solo si se miran las dos cosas juntas** | Un atípico **multivariado** |
| Sembrar costales falsos a propósito para probar el ojo del inspector | **Simular** con contaminación controlada |
| «Encontró 8 de 8 pero acusó a 3 honestos» | **Aciertos y falsas alarmas** |

Al terminar podrás explicar:
1. Por qué **los propios atípicos pueden esconderse** cuando se usa el promedio.
2. Por qué hay datos que **solo se ven raros al mirar dos medidas juntas**.
3. Por qué todo detector tiene dos errores (**no encontrar** y **acusar de más**) y por qué se prueba **sembrando casos conocidos**.

> 🔗 Si luego quieres la versión **con fórmulas, covarianza robusta, *Isolation Forest*, LOF, AUC y el equivalente en R**: [cuaderno estándar](../05_Deteccion_de_Datos_Atipicos_Simulacion_Multivariada.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb).

---
## 1. Los Costales con Piedras Engañan al Promedio 🥔

Inventamos **200 costales** de papa: 176 honestos que pesan cerca de **50 kg** y **24 «con piedras»** que pesan cerca de **62 kg**. Probamos dos reglas para encontrar los tramposos: la de **«lejos del promedio»** y la de **«lejos del valor del medio»** (la mediana).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)

# 200 costales de papa INVENTADOS: 176 de 50 kg y 24 "con piedras" que pesan más
buenos = rng.normal(50, 2, 176)
falsos = rng.normal(62, 1, 24)
pesos = np.r_[buenos, falsos]
es_falso = np.r_[np.zeros(176, bool), np.ones(24, bool)]

prom, desv = pesos.mean(), pesos.std()
med = np.median(pesos)
dispersion = np.median(np.abs(pesos - med)) * 1.4826                  # "desviación" que no se deja engañar por los extremos

regla_promedio = np.abs(pesos - prom) > 3 * desv                      # "raro = lejos del promedio"
regla_mediana = np.abs(pesos - med) > 3 * dispersion                  # "raro = lejos del valor del medio"
print(f"Promedio = {prom:.1f} kg y desviación = {desv:.1f} (la de los buenos es 2) | valor del medio = {med:.1f} kg")
print(f"Regla del promedio: marca {regla_promedio.sum()} costales (halló {(regla_promedio & es_falso).sum()} de 24 falsos)")
print(f"Regla del valor del medio: marca {regla_mediana.sum()} costales (halló {(regla_mediana & es_falso).sum()} de 24 falsos, {(regla_mediana & ~es_falso).sum()} falsas alarmas)")

plt.figure(figsize=(8, 3.6))
plt.hist(pesos, bins=40, color="#93c5fd", edgecolor="white")
plt.axvline(prom + 3 * desv, color="#d97706", label="límite de la regla del promedio")
plt.axvline(med + 3 * dispersion, color="#16a34a", label="límite de la regla del valor del medio")
plt.xlabel("peso del costal (kg)"); plt.ylabel("costales"); plt.legend(); plt.title("Los costales con piedras inflan el promedio y se esconden"); plt.show()


---
**Lección:** los **24 costales con piedras** subieron el promedio a **51.4 kg** y, peor, **estiraron la «desviación»** a **4.3** (los honestos solo varían 2 kg). Con esa regla elástica, un costal de 62 kg **ya no parece tan raro**: la regla del promedio **no marcó ninguno**. La regla del **valor del medio** (50.2 kg), que casi no se mueve por unos pocos costales extremos, **marcó los 24 y ni un honesto**. El método se vuelve **ciego precisamente por culpa de lo que debe encontrar**: cuando los tramposos son varios, **el promedio es mal juez**.

---
## 2. Raros Solo al Mirar Dos Medidas Juntas 👫

Ahora inventamos **300 adultos** con **estatura** y **peso** (más altos, en general más pesados) y **sembramos 8 casos «raros juntos»**: gente de 1,85 m que pesa solo 52 kg, o de 1,55 m que pesa 85 kg. Ninguna de las dos medidas es rara **por sí sola** (hay mucha gente de 1,85 m y de 52 kg), pero **la combinación sí**. Probamos mirar cada medida por separado y luego mirar las dos **juntas** con una distancia que tiene en cuenta **cómo van juntas** estatura y peso.

In [ ]:
from scipy import stats
from sklearn.covariance import EmpiricalCovariance

# 300 adultos INVENTADOS: estatura (cm) y peso (kg) van juntas (más altos, más pesados)
media = [165, 65]
cov = [[8 ** 2, 0.8 * 8 * 10], [0.8 * 8 * 10, 10 ** 2]]
gente = rng.multivariate_normal(media, cov, 300)
raros = np.array([[186, 52], [184, 50], [187, 54], [155, 85], [153, 88], [185, 55], [156, 84], [183, 51]])   # 8 casos "raros juntos"
datos = np.vstack([gente, raros])
es_raro = np.r_[np.zeros(300, bool), np.ones(8, bool)]

# 1) Mirando cada medida por separado
z = np.abs((datos - datos.mean(0)) / datos.std(0))
por_separado = (z > 3).any(axis=1)
print(f"Mirando cada medida por separado: hallados {(por_separado & es_raro).sum()} de 8 casos raros")

# 2) Mirando las dos medidas JUNTAS (la distancia tiene en cuenta cómo van juntas)
dist2 = EmpiricalCovariance().fit(datos).mahalanobis(datos)
corte = stats.chi2.ppf(0.99, 2)                       # valor de corte "de tabla" para dos medidas (≈ 9.2)
juntas = dist2 > corte
print(f"Mirándolas juntas (corte {corte:.1f}): hallados {(juntas & es_raro).sum()} de 8 | falsas alarmas: {(juntas & ~es_raro).sum()} de 300 personas normales")

plt.figure(figsize=(6.5, 5))
plt.scatter(*datos[~es_raro].T, s=10, color="#93c5fd", label="personas normales")
plt.scatter(*datos[es_raro].T, s=40, color="#dc2626", label="raros juntos")
plt.xlabel("estatura (cm)"); plt.ylabel("peso (kg)"); plt.legend(); plt.title("Ninguna medida es rara sola; juntas, sí"); plt.show()


---
**Lección:** mirando **cada medida por separado**, **no se halló ninguno** de los 8 casos raros (0 de 8). Mirando **las dos juntas**, se hallaron **los 8 de 8** y **ninguna falsa alarma** entre las 300 personas normales (0 de 300). Es como el niño de 1,90 m y 45 kg: alto y liviano **no es raro por separado**, pero **juntos no encajan** con cómo suelen ir estatura y peso. Por eso los datos con muchas columnas se revisan **juntas** y no solo una por una. (En el cuaderno estándar vemos que, cuando los tramposos son varios y agrupados, hay que usar una versión **robusta** de esta distancia, igual que la mediana frente al promedio.)

---
##### 🛠️ Práctica 1: ¿Dónde pongo el corte?

Con `dist2` (la distancia de cada persona), `es_raro` (qué personas son los 8 casos sembrados) y los cortes 6.0, 9.2 y 13.8:
1. Para cada corte `c`, marca como raras a las personas con `dist2 > c` y cuenta **cuántos de los 8 raros se hallan** y **cuántas falsas alarmas** hay (personas normales marcadas); guárdalo en el diccionario `resultados[c]`.
2. Comprueba con `assert` que el corte más **bajo** (6.0) halla **al menos tantos** raros como el más alto (13.8) pero también produce **al menos tantas falsas alarmas**: no existe corte gratis.

In [ ]:
# Escribe tu código aquí

# resultados = {}
# for c in (6.0, 9.2, 13.8):
#     marca = dist2 > c
#     resultados[c] = (..., ...)

# assert resultados[6.0][0] >= resultados[13.8][0] and resultados[6.0][1] >= resultados[13.8][1]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
resultados = {}
for c in (6.0, 9.2, 13.8):
    marca = dist2 > c
    resultados[c] = (int((marca & es_raro).sum()), int((marca & ~es_raro).sum()))
print("corte -> (raros hallados, falsas alarmas):", resultados)
assert resultados[6.0][0] >= resultados[13.8][0] and resultados[6.0][1] >= resultados[13.8][1]
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué la regla del promedio no encontró los 24 costales con piedras si pesan mucho más que los demás? ¿Qué le pasó al «criterio» de la regla cuando los tramposos entraron al cálculo?
2. Si un dato resulta marcado como raro, ¿lo **borrarías** de inmediato? Menciona dos explicaciones posibles (una que sí justifica corregirlo y otra que **no**) y qué le preguntarías a quien tomó el dato.
3. Si bajas el corte para no dejar escapar a ningún tramposo, ¿qué pasa con los honestos? ¿En qué casos de la vida real preferirías **acusar de más** y en cuáles **dejar pasar algún tramposo**?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>